In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv(r"C:\Users\DELL\Desktop\Kaggle\Customer Churn\train.csv")
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
ids = train['id']
train.drop('id', axis = 1, inplace= True)
train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [10]:
X = train.drop('Churn', axis=1)
y = train['Churn']

In [13]:
y = y.map({'Yes' :1, 'No':0})
y.unique()

array([0, 1])

In [22]:
numerical_features = [i for i in X.select_dtypes(exclude='object').columns]
numerical_features.remove('SeniorCitizen')
print("Numerical Features: " , len(numerical_features), numerical_features)
categorical_features = [i for i in X.select_dtypes(include='object').columns]
print("Categorical Features: " , len(categorical_features), categorical_features)

Numerical Features:  3 ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Features:  15 ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [24]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer([
    ('onehot', OneHotEncoder(), categorical_features),
    ('scaler', StandardScaler(), numerical_features)
])

X_train_preprocessed = preprocessing.fit_transform(X_train)
X_train_preprocessed

array([[ 1.        ,  0.        ,  0.        , ...,  1.21393639,
         0.55098917,  1.18769399],
       [ 1.        ,  0.        ,  1.        , ...,  0.41588312,
        -1.49315831, -0.6651811 ],
       [ 0.        ,  1.        ,  0.        , ...,  1.13413106,
         1.1046795 ,  1.84624777],
       ...,
       [ 1.        ,  0.        ,  0.        , ..., -0.38217015,
        -1.4706244 , -0.82774975],
       [ 1.        ,  0.        ,  0.        , ...,  1.01442307,
         0.08743446,  0.66375079],
       [ 0.        ,  1.        ,  0.        , ...,  1.37354704,
         1.26241687,  1.98922242]], shape=(475355, 44))

In [25]:
X_test_preprocessed = preprocessing.transform(X_test)
X_test_preprocessed

array([[ 1.        ,  0.        ,  0.        , ...,  1.37354704,
         0.75862304,  1.61228731],
       [ 1.        ,  0.        ,  0.        , ...,  0.29617513,
         0.31116399,  0.59125511],
       [ 0.        ,  1.        ,  0.        , ...,  0.6552991 ,
        -1.48511048, -0.60215337],
       ...,
       [ 0.        ,  1.        ,  0.        , ...,  0.81490975,
         0.95016127,  1.29173538],
       [ 1.        ,  0.        ,  1.        , ..., -0.94080744,
         0.14859793, -0.66749502],
       [ 1.        ,  0.        ,  1.        , ..., -0.86100211,
         0.19527531, -0.66600901]], shape=(118839, 44))

In [26]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_preprocessed)
X_test_pca = pca.transform(X_test_preprocessed)

In [27]:
X_train_pca.shape, X_test_pca.shape

((475355, 17), (118839, 17))

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic = LogisticRegression().fit(X_train_pca,y_train)

In [30]:
train_pred = logistic.predict(X_test_pca)

In [31]:
y.value_counts()

Churn
0    460377
1    133817
Name: count, dtype: int64

In [34]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score
print("Precision Score: ", precision_score(y_test, train_pred))
print("Recall Score: ", recall_score(y_test, train_pred))
print("Precision Recall Score: ", roc_auc_score(y_test, train_pred))

Precision Score:  0.6854222999840942
Recall Score:  0.6406853999405293
Precision Recall Score:  0.7773177366809843


In [38]:
params = {
    'penalty' : ['l1','l2','elasticnet'],
    'C': [0.01, 0.1, 1, 10]
}
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(logistic, param_grid=params,cv = 5 , scoring='roc_auc')
grid.fit(X_train_pca, y_train)
grid.best_score_

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
40 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\sk

np.float64(0.9041374025546437)

In [39]:
grid.best_params_

{'C': 0.01, 'penalty': 'l2'}

In [54]:
test = pd.read_csv(r"C:\Users\DELL\Desktop\Kaggle\Customer Churn\test.csv")
ids = test['id']
test.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [55]:
test = preprocessing.transform(test)
test

array([[ 1.        ,  0.        ,  0.        , ...,  1.4134497 ,
         1.59881594,  2.36295201],
       [ 1.        ,  0.        ,  0.        , ...,  1.37354704,
        -1.48350092, -0.49229534],
       [ 0.        ,  1.        ,  1.        , ..., -0.9807101 ,
        -0.33266199, -0.7907483 ],
       ...,
       [ 0.        ,  1.        ,  0.        , ..., -0.06294884,
         1.28495078,  0.27034229],
       [ 1.        ,  0.        ,  1.        , ..., -0.46197548,
        -1.46901483, -0.84267346],
       [ 1.        ,  0.        ,  1.        , ..., -0.50187814,
         0.77954739, -0.13754413]], shape=(254655, 44))

In [56]:
test = pca.transform(test)
test

array([[ 3.20434658,  2.14019488, -0.66197314, ..., -0.69307999,
         0.90107281, -0.00652791],
       [-2.54242608,  2.16436557, -0.40589502, ..., -0.67057131,
        -0.20727935, -0.00612037],
       [-0.50005606, -1.57465153,  1.39892291, ...,  0.13327756,
        -0.6468124 ,  0.00393574],
       ...,
       [ 1.81696521,  0.13943077, -0.55206156, ..., -0.79353444,
        -0.65423026, -0.01018751],
       [-2.79125899,  0.53992163, -0.81276257, ...,  0.11388299,
        -0.32208363, -0.0167426 ],
       [ 1.05930565, -1.29013005, -0.68122151, ...,  0.45690724,
        -0.78079651,  0.99572879]], shape=(254655, 17))

In [57]:
grid_preds = grid.predict(test)

In [58]:
grid_preds

array([0, 0, 0, ..., 0, 0, 0], shape=(254655,))

In [71]:
answer = pd.concat([ids, pd.DataFrame(grid_preds, columns=['Churn'])],axis =1)

In [72]:
answer.head()

,id,Churn
0,594194,0
1,594195,0
2,594196,0
3,594197,0
4,594198,1


In [74]:
answer.to_csv(r"C:\Users\DELL\Desktop\Kaggle\Customer Churn\submissions\logistic_predictions.csv",index=False)

In [70]:
pd.DataFrame(grid_preds, columns=['Churn'])

,Churn
0,0
1,0
2,0
3,0
4,1
...,...
254650,0
254651,1
254652,0
254653,0


In [81]:
X_train = pd.DataFrame(X_train_pca)
X_train.to_csv(r"C:\Users\DELL\Desktop\Kaggle\Customer Churn\X_train.csv")

In [82]:
X_test = pd.DataFrame(X_test_pca)
X_test.to_csv(r"C:\Users\DELL\Desktop\Kaggle\Customer Churn\X_test.csv")